# D\&R Replication Guide

**Debate & Reflect — Knowledge Distillation on MMLU-Pro Computer Science**

This notebook walks through the full replication: environment setup → multi-agent debate → SFT + T-DPO training → evaluation.

**This notebook requires that Ubuntu and some Ubuntu packages are installed** Should you not have it installed, kindly follow the steps below before proceeding (in Powershell):
 
- Install Ubuntu: wsl --install -d Ubuntu 
- Restart your machine.
- After reboot an Ubuntu terminal window should open automatically. If it does not, kindly launch Ubuntu. It should finish unpacking and ask you to create a UNIX username and password.
- This should result in you being in Ubuntu. If it does not, run: wsl -d Ubuntu
- Update Ubuntu packages: sudo apt update && sudo apt upgrade -y
- Install tools required by the notebook: sudo apt install -y build-essential cmake git wget curl ca-certificates

This should be enough to get Ubuntu set up.

**Run this notebook inside WSL (Ubuntu).** Open a terminal, activate your conda environment, then launch Jupyter by following these commands in Powershell:


 wsl -d Ubuntu
- cd 'your path'. for example:cd /mnt/c/Users/claudio/Downloads/D-R-master/nlp assignment
- bash replications/setup_environment.sh
- NOTE: should bash replications/setup_environment.sh fail due to some 'invalid' characters in the sript, it is due to the difference between how the script gets pushed to git (CRLF vs LF). Please run the following commands, save the script and attempt running again:
    - sed -i 's/\r$//' replications/setup_environment.sh
     - dos2unix replications/setup_environment.sh
- (run if dr_env is not activated automatically from setup_environment.sh) conda activate dr_env 
- conda install jupyter -y
- jupyter notebook replication_guide.ipynb

once jupyter is launched, navigate to this notebook on there and follow each cell.
---

## Table of Contents
1. [Prerequisites](#1-prerequisites)
2. [Environment Setup](#2-environment-setup)
3. [API Keys](#3-api-keys)
4. [Data Preparation](#4-data-preparation)
5. [Multi-Agent Debate (10 questions)](#5-multi-agent-debate)
6. [Build Training Data](#6-build-training-data)
7. [SFT Training](#7-sft-training)
8. [T-DPO Training](#8-t-dpo-training)
9. [Merge & Convert to GGUF](#9-merge--convert-to-gguf)
10. [Evaluate Both Models](#10-evaluate-both-models)
11. [Results Summary](#11-results-summary)
12. [Robustness Test (5 questions)](#12-robustness-test-5-questions)

---
## 1. Prerequisites

Before starting, confirm you have:

| Requirement | Notes |
|---|---|
| Windows 10/11 with WSL2 | Install via `wsl --install` in admin PowerShell |
| Ubuntu distro | `wsl --install -d Ubuntu` |
| 16 GB RAM RECOMMENDED 
| API keys | OpenAI, Anthropic, Google Gemini |

In [23]:
# Check system: free RAM and number of CPU threads
!free -h
!echo "CPU threads: $(nproc)"

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.3Gi       8.1Gi        64Ki       3.3Gi        11Gi
Swap:          4.0Gi        24Mi       4.0Gi
CPU threads: 4


---
## 2. Environment Setup

Run this **once**. It installs Miniconda (if needed) and creates the `dr_env` conda environment with all dependencies.

In [24]:
# Run from the repo root
import os
#replace the path with the one shown in the terminal after activating Ubuntu though 'wsl -d Ubuntu'
os.chdir('/mnt/c/Users/claudio/Downloads/D-R-master/nlp assignment')
!pwd

/mnt/c/Users/claudio/Downloads/D-R-master/nlp assignment


In [25]:
import torch
import transformers
import peft
import trl
import datasets
import openai
import anthropic
import llama_cpp

print(f'torch: {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
print(f'trl: {trl.__version__}')

torch: 2.4.1+cpu
transformers: 4.50.0
peft: 0.15.0
trl: 0.16.0


---
## 3. Download Models

Two model files are required before anything can run:

| Model | Use | Size | How obtained |
|---|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` (HuggingFace) | SFT + T-DPO training | ~3 GB | Downloaded below (or auto on first train run) |
| `qwen2.5-1.5b-instruct-q4_k_m.gguf` | Debate student + base evaluation | ~1.1 GB | Downloaded below |

In [26]:
# Download the base Qwen2.5-1.5B-Instruct GGUF (Q4) for llama.cpp inference
import os
os.makedirs('./models/Qwen2.5-1.5B-Instruct-GGUF', exist_ok=True)
gguf_path = './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf'

if os.path.exists(gguf_path):
    print(f'Already exists: {gguf_path}')
else:
    print('Downloading GGUF...')
    !wget -q --show-progress \
        "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf" \
        -O {gguf_path}
    print('Done.')


Already exists: ./models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf


In [27]:
# Pre-download the HuggingFace weights for training (caches to ~/.cache/huggingface/)
print('Downloading Qwen2.5-1.5B-Instruct HF weights ...')

from transformers import AutoModelForCausalLM, AutoTokenizer

AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct', torch_dtype='float32')
print('Download complete.')


Download complete.


---
## 3. API Keys

Set your API keys below. These are needed for the debate phase (GPT-4o, Claude, Gemini).

In [ ]:
import os

os.environ['OPENAI_API_KEY']    = ''    
os.environ['ANTHROPIC_API_KEY'] = ''  
os.environ['GEMINI_API_KEY']    = ''     

# Check keys are set.
for key in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY']:
    val = os.environ.get(key, '')
    status = 'SET' if val and not val.endswith('...') else 'NOT SET'
    print(f'{key}: {status}')

OPENAI_API_KEY: SET
ANTHROPIC_API_KEY: SET
GEMINI_API_KEY: SET


---
## 4. Data Preparation

Downloads MMLU-Pro Computer Science questions.

In [10]:
!python get_and_split_data.py

/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
410 original test data
DatasetDict({
    train: Dataset({
        features: ['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src'],
        num_rows: 205
    })
    test: Dataset({
        features: ['question_id', 'question', 'options', 'answer', 'answer_index', 'cot_content', 'category', 'src'],
        num_rows: 205
    })
})
205
205


---
## 5. Multi-Agent Debate

Runs a 4-agent debate across 4 rounds on 10 training questions.

| Agent | Model |
|---|---|
| Teacher 1 | GPT-4o (`gpt-4o-2024-08-06`) |
| Teacher 2 | Claude (`claude-opus-4-6`) |
| Teacher 3 | Gemini (`gemini-2.5-flash`) |
| Student | Qwen2.5-1.5B-Instruct Q4 (local, llama.cpp) |

The student model is served locally on port 58000 during the debate.

In [11]:
# Start the student model server
import subprocess, time, requests

server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/llama_debate.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for server..')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)
else:
    print('ERROR: server did not start. Check /tmp/llama_debate.log')

Waiting for server..
Server ready.


In [13]:
# Run the debate (10 questions)
# Should you wish to run the debate on 1 question, change --num_questions=1 for a faster run or to whichever number you wish.
!python debate_teachers_student_w_critique.py --dataset MMLUProComp --num_questions=10, 

  Using cached fastuuid-0.14.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.1 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (6.7 kB)
  Using cached importlib_metadata-8.5.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jsonschema-4.23.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached pydantic_core-2.41.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 27.9 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.0 MB/s  0:00:00
Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
Using cached pydantic_

In [14]:
# Stop the student model server
server.terminate()
server.wait()
print('Server stopped.')

Server stopped.


In [15]:
# Check debate output
!ls -lh data/MAG_new_mistral/MMLUProComp_1000.jsonl
!echo "Lines (= debate entries):"
!wc -l data/MAG_new_mistral/MMLUProComp_1000.jsonl

-rwxrwxrwx 1 claudiozammit claudiozammit 60K Apr 27 01:28 data/MAG_new_mistral/MMLUProComp_1000.jsonl
Lines (= debate entries):
1 data/MAG_new_mistral/MMLUProComp_1000.jsonl


---
## 6. Build Training Data

Converts the debate output into two formats:
- **SFT**: chosen (correct) reasoning chains
- **T-DPO**: preference pairs (chosen vs rejected answers)

In [16]:
# Build SFT data
!python mag2sft.py
!echo "SFT examples:"
!wc -l data/mag_new_sft_w_mistral/MMLUProComp_mag_new_974_sft_w_mistral.jsonl

SFT examples:
11 data/mag_new_sft_w_mistral/MMLUProComp_mag_new_974_sft_w_mistral.jsonl


In [18]:
# Build T-DPO preference pairs
!python mag2preference.py
!echo "Preference pairs:"
!wc -l data/mag_new_preference_pair_w_mistral/MMLUProComp_mag_new_974_preference_pair_w_mistral.jsonl

  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.1 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.83.14 requires tokenizers==0.22.2, but you have tokenizers 0.21.4 which is incompatible.
/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
Preference pairs:
13 data/mag_new_preference_pair_w_mistral/MMLUProComp_mag_new_974_preference_pair_w_mistral.jsonl


---
## 7. SFT Training

Fine-tunes Qwen2.5-1.5B-Instruct using Supervised Fine-Tuning on the D&R chosen examples.

**Hyperparameters:**

| Parameter | Value |
|---|---|
| Base model | `Qwen/Qwen2.5-1.5B-Instruct` |
| Epochs | 2 |
| Learning rate | 2e-4 |
| Batch size | 1 (grad accum steps: 4) |
| Max sequence length | 512 tokens |
| LoRA rank (r) | 16 |
| LoRA alpha | 32 |
| Optimizer | Adafactor |
| Dtype | float32 |
| Seed | 42 |

> ⏱ ~2–4 hours on CPU.

In [19]:
%%bash
CUDA_VISIBLE_DEVICES="" TOKENIZERS_PARALLELISM=false python sft.py \
    --dataset_name ./data/mag_new_sft_w_mistral/MMLUProComp_mag_new_974_sft_w_mistral.jsonl \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --learning_rate 2.0e-4 \
    --num_train_epochs 2 \
    --packing false \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 4 \
    --max_seq_length 512 \
    --use_peft \
    --lora_r 16 \
    --lora_alpha 32 \
    --lora_task_type CAUSAL_LM \
    --logging_steps 0.1 \
    --eval_strategy no \
    --save_strategy steps \
    --save_steps 0.5 \
    --output_dir ./tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch \
    --optim adafactor \
    --torch_dtype float32 \
    --bf16 false \
    --fp16 false \
    --dataloader_num_workers 1 \
    --seed 42


/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


04/27/2026 01:31:54 - WARNING - __main__ - Process rank: 0, device: cpu, n_gpu: 0, distributed training: False, 16-bits training: False
04/27/2026 01:31:54 - INFO - __main__ - Training/evaluation parameters SFTConfig(
_n_gpu=0,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=False,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
chars_per_token=<CHARS_PER_TOKEN>,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=1,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_batch_size=None,
dataset_kwargs=None,
dataset_num_proc=None,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=

[INFO|configuration_utils.py:699] 2026-04-27 01:31:54,355 >> loading configuration file config.json from cache at /home/claudiozammit/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:771] 2026-04-27 01:31:54,359 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.50.0",
  "use_cache": true,
  "use_sliding_window": false,


04/27/2026 01:32:29 - INFO - datasets.builder - Using custom data configuration default-5c6d9913c69a238a


Loading Dataset Infos from /home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/datasets/packaged_modules/json


...

04/27/2026 01:32:30 - INFO - datasets.arrow_dataset - Caching processed dataset at /home/claudiozammit/.cache/huggingface/datasets/json/default-5c6d9913c69a238a/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-6a54115f2d2a4862.arrow


Truncating train dataset: 100%|██████████| 11/11 [00:00<00:00, 327.46 examples/s]
[WARNING|trainer.py:783] 2026-04-27 01:32:31,817 >> No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
[INFO|trainer.py:930] 2026-04-27 01:32:32,929 >> The following columns in the training set don't have a corresponding argument in `PeftModelForCausalLM.forward` and have been ignored: text. If text are not expected by `PeftModelForCausalLM.forward`,  you can safely ignore this message.
[INFO|trainer.py:2409] 2026-04-27 01:32:32,962 >> ***** Running training *****
[INFO|trainer.py:2410] 2026-04-27 01:32:32,962 >>   Num examples = 11
[INFO|trainer.py:2411] 2026-04-27 01:32:32,962 >>   Num Epochs = 2
[INFO|trainer.py:2412] 2026-04-27 01:32:32,963 >>   Instantaneous batch size per device = 1
[INFO|trainer

{'loss': 1.4689, 'grad_norm': 0.781408429145813, 'learning_rate': 0.00015000000000000001, 'num_tokens': 1535.0, 'mean_token_accuracy': 0.713158056139946, 'epoch': 0.36}
{'loss': 1.5191, 'grad_norm': 0.6721627116203308, 'learning_rate': 0.0001, 'num_tokens': 3016.0, 'mean_token_accuracy': 0.677661120891571, 'epoch': 0.73}
{'loss': 1.3501, 'grad_norm': 0.7202438116073608, 'learning_rate': 5e-05, 'num_tokens': 4166.0, 'mean_token_accuracy': 0.7110471328099569, 'epoch': 1.0}


100%|██████████| 4/4 [04:57<00:00, 73.26s/it][INFO|trainer.py:3966] 2026-04-27 01:37:30,558 >> Saving model checkpoint to ./tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch/checkpoint-4
[INFO|configuration_utils.py:699] 2026-04-27 01:37:31,216 >> loading configuration file config.json from cache at /home/claudiozammit/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:771] 2026-04-27 01:37:31,229 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope

{'loss': 1.3902, 'grad_norm': 0.6880539655685425, 'learning_rate': 0.0, 'num_tokens': 5695.0, 'mean_token_accuracy': 0.7044280767440796, 'epoch': 1.36}
{'train_runtime': 298.9631, 'train_samples_per_second': 0.074, 'train_steps_per_second': 0.013, 'train_loss': 1.4320835769176483, 'epoch': 1.36}


In [20]:
# Verify SFT adapter was saved
!ls tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch/

README.md		   checkpoint-2		    tokenizer.json
adapter_config.json	   checkpoint-4		    tokenizer_config.json
adapter_model.safetensors  merges.txt		    training_args.bin
added_tokens.json	   special_tokens_map.json  vocab.json


---
## 8. T-DPO Training

Applies Teacher-guided DPO on top of the SFT adapter using the preference pairs.

**Hyperparameters:**

| Parameter | Value |
|---|---|
| Epochs | 3 |
| Learning rate | 5e-6 |
| Batch size | 1 (grad accum steps: 8) |
| Max length | 512 tokens |
| Max prompt length | 256 tokens |
| Warmup ratio | 0.1 |
| Optimizer | Adafactor |
| Gradient checkpointing | Yes |
| Seed | 42 |

> ⏱ ~3–6 hours on CPU for 10 questions D&R .

In [29]:
%%bash
# change the directory to your path eg: 
cd  /mnt/c/Users/claudio/Downloads/D-R-master/nlp assignment
CUDA_VISIBLE_DEVICES="" python tdpo_after_sft.py \
    --dataset_name ./data/mag_new_preference_pair_w_mistral/MMLUProComp_mag_new_974_preference_pair_w_mistral.jsonl \
    --model_name_or_path Qwen/Qwen2.5-1.5B-Instruct \
    --learning_rate 5.0e-6 \
    --num_train_epochs 3 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --max_length 512 \
    --max_prompt_length 256 \
    --truncation_mode keep_end \
    --no_remove_unused_columns \
    --use_peft \
    --warmup_ratio 0.1 \
    --logging_steps 0.1 \
    --eval_strategy no \
    --save_strategy steps \
    --save_steps 0.1 \
    --output_dir ./tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch \
    --torch_dtype float32 \
    --dataloader_num_workers 0 \
    --precompute_ref_log_probs \
    --model_adapter_name tdpo \
    --ref_adapter_name reference \
    --sft_adapter_path ./tmp_sft/sft_chosen_MMLUProComp_mag_new_974_w_qwen_2epoch \
    --optim adafactor \
    --gradient_checkpointing \
    --seed 42


bash: line 2: cd: too many arguments
/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


04/27/2026 20:51:52 - WARNING - __main__ - Process rank: 0, device: cpu, n_gpu: 0, distributed training: False, 16-bits training: False


[INFO|configuration_utils.py:699] 2026-04-27 20:51:52,431 >> loading configuration file config.json from cache at /home/claudiozammit/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:771] 2026-04-27 20:51:52,433 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "max_position_embeddings": 32768,
  "max_window_layers": 21,
  "model_type": "qwen2",
  "num_attention_heads": 12,
  "num_hidden_layers": 28,
  "num_key_value_heads": 2,
  "rms_norm_eps": 1e-06,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "sliding_window": 32768,
  "tie_word_embeddings": true,
  "torch_dtype": "float32",
  "transformers_version": "4.50.0",
  "use_cache": false,
  "use_sliding_window": false,

04/27/2026 20:52:11 - INFO - datasets.builder - Using custom data configuration default-f72be3d129afc946
04/27/2026 20:52:11 - INFO - datasets.info - Loading Dataset Infos from /home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/datasets/packaged_modules/json


Loading Dataset Infos from /home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/datasets/packaged_modules/json
Overwrite dataset info from restored data version if exists.


...

04/27/2026 20:52:11 - INFO - datasets.arrow_dataset - Loading cached processed dataset at /home/claudiozammit/.cache/huggingface/datasets/json/default-f72be3d129afc946/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-e90f649f52163c98.arrow


Loading cached processed dataset at /home/claudiozammit/.cache/huggingface/datasets/json/default-f72be3d129afc946/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-22dafdb63bf2cc0b.arrow


04/27/2026 20:52:11 - INFO - datasets.arrow_dataset - Loading cached processed dataset at /home/claudiozammit/.cache/huggingface/datasets/json/default-f72be3d129afc946/0.0.0/f4e89e8750d5d5ffbef2c078bf0ddfedef29dc2faff52a6255cf513c05eb1092/cache-22dafdb63bf2cc0b.arrow


[WARNING|trainer.py:783] 2026-04-27 20:52:12,171 >> No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
Train dataset reference log probs: 100%|██████████| 13/13 [04:39<00:00, 21.48s/it]
[INFO|trainer.py:2409] 2026-04-27 20:56:51,675 >> ***** Running training *****
[INFO|trainer.py:2410] 2026-04-27 20:56:51,676 >>   Num examples = 13
[INFO|trainer.py:2411] 2026-04-27 20:56:51,676 >>   Num Epochs = 3
[INFO|trainer.py:2412] 2026-04-27 20:56:51,676 >>   Instantaneous batch size per device = 1
[INFO|trainer.py:2415] 2026-04-27 20:56:51,676 >>   Total train batch size (w. parallel, distributed & accumulation) = 8
[INFO|trainer.py:2416] 2026-04-27 20:56:51,676 >>   Gradient Accumulation steps = 8
[INFO|trainer.py:2417] 2026-04-27 20:56:51,676 >>   Total optimization steps = 3
[INFO|trainer

{'loss': 0.6931, 'grad_norm': 4.094504356384277, 'learning_rate': 5e-06, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -192.11119079589844, 'logps/rejected': -48.527076721191406, 'logits/chosen': 0.2772684097290039, 'logits/rejected': 0.7867841720581055, 'epoch': 0.62}
{'loss': 0.4332, 'grad_norm': 3.3962979316711426, 'learning_rate': 2.5e-06, 'rewards/chosen': 0.0, 'rewards/rejected': 0.0, 'rewards/accuracies': 0.0, 'rewards/margins': 0.0, 'logps/chosen': -218.4244384765625, 'logps/rejected': -48.478126525878906, 'logits/chosen': 0.34204739332199097, 'logits/rejected': 0.7312917709350586, 'epoch': 1.0}
{'loss': 0.6786, 'grad_norm': 5.810309886932373, 'learning_rate': 0.0, 'rewards/chosen': 0.019605351611971855, 'rewards/rejected': -0.009756803512573242, 'rewards/accuracies': 1.0, 'rewards/margins': 0.029362153261899948, 'logps/chosen': -196.04623413085938, 'logps/rejected': -45.67719650268555, 'logits/chosen': 0.1580

In [31]:
# Verify T-DPO adapter was saved
!ls tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch/tdpo/

adapter_config.json  adapter_model.safetensors


---
## 9. Merge & Convert to GGUF

Merges the LoRA adapter back into the base model weights, then converts to GGUF format for llama.cpp inference.

In [32]:
# Merge adapter into base model
#This may take some time. It will print 'Done.' when done
!python replications/merge_adapter.py \
    --adapter_path ./tmp_tdpo/tdpo_after_sft_MMLUProComp_mag_new_974_w_qwen_3epoch \
    --base_model   Qwen/Qwen2.5-1.5B-Instruct \
    --output_dir   ./models/qwen-distilled-merged \
    --adapter_name tdpo

/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Done.


In [33]:
# Convert to GGUF (f16)
LLAMA_CPP_DIR="${LLAMA_CPP_DIR:-./llama.cpp}"
!python {LLAMA_CPP_DIR}/convert_hf_to_gguf.py ./models/qwen-distilled-merged \
    --outfile ./models/qwen-distilled.gguf \
    --outtype f16

/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
INFO:hf-to-gguf:Loading model: qwen-distilled-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00002.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00002.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float32 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float32 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float32 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float32 --> F16,

In [35]:
# Verify GGUF was created
!ls -lh models/qwen-distilled.gguf

-rwxrwxrwx 1 claudiozammit claudiozammit 2.9G Apr 27 21:21 models/qwen-distilled.gguf


---
## 10. Evaluate Both Models

Evaluates both the base model and the distilled model on the 137-question test set.

**Evaluation settings:**

| Parameter | Value |
|---|---|
| Dataset | MMLU-Pro Computer Science |
| Test questions | 137 |
| Temperature | 0.3 |
| Top-p | 0.9 |
| Max new tokens | 700 |
| Server port | 58000 |

In [36]:
# Evaluate the BASE model
import os, subprocess, time, requests

base_server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/Qwen2.5-1.5B-Instruct-GGUF/qwen2.5-1.5b-instruct-q4_k_m.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/eval_base.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for base model server...')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)


Waiting for base model server...
Server ready.


In [ ]:
!python test.py \
    --dataset MMLUProComp \
    --model_name qwen2.5-1.5b-instruct-q4_k_m \
    --num_proc 1 \
    --max_new_tokens 700 \
    --temperature 0.3 \
    --top_p 0.9 \
    --port 58000 \
    --num_questions 205

/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
26-04-27 21:28 === evaluating qwen2.5-1.5b-instruct-q4_k_m on MMLUProComp dataset ===
{'max_tokens': 700, 'max_completion_tokens': 700, 'top_p': 0.9, 'temperature': 0.3}
5 5
Map: 100%|█████████████████████████████████| 5/5 [01:16<00:00, 15.32s/ examples]
5
token cnt:  461.0
26-04-27 21:29 | samples evaluated: 5 | accuracy: 0.2
Saving the dataset (1/1 shards): 100%|████| 5/5 [00:00<00:00, 680.61 examples/s]


In [39]:
base_server.terminate(); base_server.wait(); print('Base model server stopped.')

Base model server stopped.


In [40]:
# Evaluate the DISTILLED model
import os, subprocess, time, requests

dist_server = subprocess.Popen([
    'python', '-m', 'llama_cpp.server',
    '--model', './models/qwen-distilled.gguf',
    '--host', '0.0.0.0', '--port', '58000',
    '--n_ctx', '4096', '--n_threads', str(os.cpu_count())
], stdout=open('/tmp/eval_distilled.log', 'w'), stderr=subprocess.STDOUT)

print('Waiting for distilled model server...')
for _ in range(120):
    try:
        if requests.get('http://localhost:58000/v1/models', timeout=2).status_code == 200:
            print('Server ready.')
            break
    except:
        pass
    time.sleep(5)


Waiting for distilled model server...
Server ready.


In [ ]:
!python test.py \
    --dataset MMLUProComp \
    --model_name distilled-qwen \
    --num_proc 1 \
    --max_new_tokens 700 \
    --temperature 0.3 \
    --top_p 0.9 \
    --port 58000\
    --num_questions 205

/home/claudiozammit/miniconda3/envs/dr_env/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
26-04-27 21:38 === evaluating distilled-qwen on MMLUProComp dataset ===
{'max_tokens': 700, 'max_completion_tokens': 700, 'top_p': 0.9, 'temperature': 0.3}
20 20
Map: 100%|███████████████████████████████| 20/20 [20:06<00:00, 60.31s/ examples]
20
token cnt:  619.9
26-04-27 21:59 | samples evaluated: 20 | accuracy: 0.35
Saving the dataset (1/1 shards): 100%|█| 20/20 [00:00<00:00, 2246.01 examples/s]


In [ ]:
dist_server.terminate(); dist_server.wait(); print('Distilled model server stopped.')

---
## 11. Results Summary

Results obtained in this experiment (2026-04-26, seed=42, 10-question debate):

| Run | Model | Debate Qs | Correct | Total | Accuracy |
|---|---|---|---|---|---|
| Base | Qwen2.5-1.5B-Instruct Q4 | — | 24 | 137 | **17.52%** |
| 10q distilled | Qwen2.5-1.5B-Instruct (SFT+T-DPO) | 10 | 31 | 137 | **22.63%** |

The distilled model (10q) improves over the base by **+5.11 percentage points**.

Results may vary slightly between runs due to temperature=0.3 sampling over 137 questions.